# Implementasi Webinar Pertemuan 4
## Introduction to Machine Learning — Prediksi Konsumsi Energi Gedung

**Platform:** Python Google Colab  
**Tingkat kompleksitas:** Menengah  
**Studi kasus:** Prediksi konsumsi energi listrik harian gedung berdasarkan suhu, jumlah penghuni, jam operasional, dan jumlah peralatan listrik.

> **Catatan:** Dataset merupakan data simulasi untuk pembelajaran, bukan data gedung nyata. Prediksi tidak boleh digunakan sebagai dasar tunggal untuk keputusan operasional atau keselamatan.

## 1. Latar Belakang dan Tujuan

Pengelola gedung perlu memahami perkiraan konsumsi listrik untuk membantu perencanaan energi dan mengidentifikasi pola penggunaan. *Machine Learning* dapat mempelajari pola dari data historis untuk memperkirakan konsumsi energi.

Studi kasus ini menggunakan **supervised learning — regression**, karena target yang diprediksi berupa angka konsumsi listrik dalam kWh.

Tujuan implementasi:
1. Membuat dan memahami dataset simulasi konsumsi energi.
2. Memeriksa kualitas data dan melakukan prapemrosesan.
3. Membagi data menjadi data latih dan data uji.
4. Melatih dan membandingkan model *Linear Regression* dan *Random Forest Regressor*.
5. Mengevaluasi model dengan MAE, RMSE, dan R².
6. Memeriksa indikasi *overfitting*, memahami fitur yang berpengaruh, dan membuat prediksi untuk kondisi gedung baru.
7. Membahas keterbatasan serta etika penggunaan model.

## 2. Memahami Konsep Utama

- **AI:** bidang luas untuk membuat sistem yang melakukan tugas yang membutuhkan kecerdasan.
- **Machine Learning:** bagian dari AI yang mempelajari pola dari data.
- **Supervised learning:** belajar dari data yang memiliki target/label.
- **Regression:** memprediksi nilai numerik, misalnya konsumsi listrik dalam kWh.
- **Training data:** data untuk melatih model.
- **Testing data:** data terpisah untuk mengukur kinerja pada contoh yang belum dilihat model.
- **Feature:** variabel masukan seperti suhu atau jam operasional.
- **Target:** nilai yang ingin diprediksi, yaitu konsumsi energi.

Alur notebook: simulasi data → pemeriksaan kualitas → eksplorasi → pemisahan data → pelatihan → evaluasi → interpretasi → prediksi baru.

## 3. Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Library berhasil diimpor.")

## 4. Membuat Dataset Simulasi

Dataset dibuat untuk 600 pengamatan harian dengan fitur:
- `suhu_rata_rata_c`: suhu rata-rata harian dalam derajat Celsius.
- `jumlah_penghuni`: perkiraan jumlah orang di gedung.
- `jam_operasional`: durasi operasional gedung per hari.
- `jumlah_peralatan`: jumlah peralatan listrik yang digunakan.
- `konsumsi_energi_kwh`: target prediksi konsumsi listrik harian.

Target dibentuk dari kombinasi fitur dan variasi acak. Rumus ini hanya asumsi untuk latihan, bukan model fisik konsumsi energi yang tervalidasi.

In [ ]:
n = 600

suhu = np.clip(np.random.normal(loc=29, scale=3.5, size=n), 20, 38)
penghuni = np.clip(np.random.normal(loc=120, scale=45, size=n), 10, 250).round().astype(int)
jam_operasional = np.clip(np.random.normal(loc=10, scale=2.5, size=n), 4, 16)
jumlah_peralatan = np.clip(np.random.normal(loc=55, scale=15, size=n), 20, 100).round().astype(int)
noise = np.random.normal(loc=0, scale=18, size=n)

konsumsi = (
    35
    + 3.2 * suhu
    + 0.42 * penghuni
    + 5.5 * jam_operasional
    + 1.15 * jumlah_peralatan
    + 2.0 * np.maximum(suhu - 27, 0)
    + noise
)
konsumsi = np.clip(konsumsi, 50, None)

df = pd.DataFrame({
    "suhu_rata_rata_c": np.round(suhu, 1),
    "jumlah_penghuni": penghuni,
    "jam_operasional": np.round(jam_operasional, 1),
    "jumlah_peralatan": jumlah_peralatan,
    "konsumsi_energi_kwh": np.round(konsumsi, 1)
})

df.head()

### 4.1 Melihat ukuran dan ringkasan dataset

In [ ]:
print(f"Jumlah baris: {df.shape[0]}")
print(f"Jumlah kolom: {df.shape[1]}")
display(df.describe().round(2))

## 5. Pemeriksaan Kualitas Data

Kualitas data memengaruhi kualitas model. Kita memeriksa tipe data, nilai kosong, duplikasi, dan rentang nilai. Untuk mendemonstrasikan penanganan data kosong, beberapa nilai fitur sengaja dibuat kosong pada salinan dataset.

In [ ]:
print("Tipe data:")
display(df.dtypes.to_frame("tipe_data"))

print("Jumlah nilai kosong per kolom:")
display(df.isna().sum().to_frame("jumlah_kosong"))

print("Jumlah baris duplikat:", df.duplicated().sum())

data = df.copy()
missing_rows = np.random.choice(data.index, size=18, replace=False)
data.loc[missing_rows[:6], "suhu_rata_rata_c"] = np.nan
data.loc[missing_rows[6:12], "jumlah_penghuni"] = np.nan
data.loc[missing_rows[12:], "jumlah_peralatan"] = np.nan

print("\nSetelah simulasi nilai kosong:")
display(data.isna().sum().to_frame("jumlah_kosong"))

### 5.1 Visualisasi distribusi target dan hubungan fitur

Grafik membantu memahami distribusi konsumsi energi dan hubungan awal antara jumlah penghuni dengan konsumsi listrik. Pola pada data simulasi tidak membuktikan hubungan sebab-akibat.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df["konsumsi_energi_kwh"], bins=24, edgecolor="black")
axes[0].set_title("Distribusi Konsumsi Energi Harian")
axes[0].set_xlabel("Konsumsi energi (kWh)")
axes[0].set_ylabel("Jumlah pengamatan")

axes[1].scatter(df["jumlah_penghuni"], df["konsumsi_energi_kwh"], alpha=0.6)
axes[1].set_title("Jumlah Penghuni dan Konsumsi Energi")
axes[1].set_xlabel("Jumlah penghuni")
axes[1].set_ylabel("Konsumsi energi (kWh)")

plt.tight_layout()
plt.show()

## 6. Menentukan Fitur dan Target serta Membagi Data

Target (`y`) adalah `konsumsi_energi_kwh`, sedangkan fitur (`X`) adalah empat variabel prediktor. Data dibagi menjadi 80% data latih dan 20% data uji. Data uji tidak digunakan untuk melatih model.

In [ ]:
features = [
    "suhu_rata_rata_c",
    "jumlah_penghuni",
    "jam_operasional",
    "jumlah_peralatan"
]
target = "konsumsi_energi_kwh"

X = data[features]
y = data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

print("Ukuran X_train:", X_train.shape)
print("Ukuran X_test :", X_test.shape)
print("Proporsi data latih:", round(len(X_train) / len(data) * 100, 1), "%")
print("Proporsi data uji  :", round(len(X_test) / len(data) * 100, 1), "%")

## 7. Melatih Dua Model

Kita membandingkan:
1. **Linear Regression** — mempelajari hubungan linear antara fitur dan target. Fitur numerik distandardisasi; nilai kosong diisi dengan median.
2. **Random Forest Regressor** — gabungan banyak pohon keputusan yang dapat menangkap pola nonlinier. Model ini tidak membutuhkan standardisasi, tetapi tetap menggunakan imputasi median.

Langkah prapemrosesan diletakkan dalam *pipeline* untuk mengurangi risiko kebocoran data (*data leakage*).

In [ ]:
linear_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

forest_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestRegressor(
        n_estimators=250,
        min_samples_leaf=3,
        random_state=RANDOM_STATE
    ))
])

linear_model.fit(X_train, y_train)
forest_model.fit(X_train, y_train)

print("Kedua model selesai dilatih.")

## 8. Evaluasi Model

Metrik yang digunakan:
- **MAE (*Mean Absolute Error*):** rata-rata selisih absolut prediksi dan nilai sebenarnya. Semakin kecil semakin baik.
- **RMSE (*Root Mean Squared Error*):** memberi penalti lebih besar pada kesalahan yang besar. Semakin kecil semakin baik.
- **R²:** mengukur seberapa baik model menjelaskan variasi target pada data uji. Nilai mendekati 1 umumnya lebih baik.

In [ ]:
def evaluate_regression(model, X_part, y_part, model_name, split_name):
    pred = model.predict(X_part)
    return {
        "Model": model_name,
        "Data": split_name,
        "MAE": mean_absolute_error(y_part, pred),
        "RMSE": np.sqrt(mean_squared_error(y_part, pred)),
        "R2": r2_score(y_part, pred)
    }

results = []
for name, model in [
    ("Linear Regression", linear_model),
    ("Random Forest", forest_model)
]:
    results.append(evaluate_regression(model, X_test, y_test, name, "Test"))

results_df = pd.DataFrame(results).sort_values("RMSE")
display(results_df.round(3))

### 8.1 Membandingkan nilai aktual dan prediksi

Titik yang semakin dekat dengan garis diagonal ideal menunjukkan prediksi yang semakin mendekati konsumsi aktual.

In [ ]:
pred_linear = linear_model.predict(X_test)
pred_forest = forest_model.predict(X_test)

upper = max(y_test.max(), pred_linear.max(), pred_forest.max()) * 1.05
lower = min(y_test.min(), pred_linear.min(), pred_forest.min()) * 0.95

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, pred, title in [
    (axes[0], pred_linear, "Linear Regression"),
    (axes[1], pred_forest, "Random Forest")
]:
    ax.scatter(y_test, pred, alpha=0.7)
    ax.plot([lower, upper], [lower, upper], linestyle="--")
    ax.set_xlabel("Konsumsi aktual (kWh)")
    ax.set_ylabel("Konsumsi prediksi (kWh)")
    ax.set_title(title)

plt.tight_layout()
plt.show()

## 9. Memeriksa Indikasi Overfitting

Kita membandingkan kinerja pada data latih dan data uji. Jika hasil data latih jauh lebih baik daripada data uji, model mungkin terlalu menyesuaikan diri dengan data latih. Perbedaan metrik saja bukan diagnosis mutlak.

In [ ]:
overfit_rows = []
for name, model in [
    ("Linear Regression", linear_model),
    ("Random Forest", forest_model)
]:
    for split_name, X_part, y_part in [
        ("Train", X_train, y_train),
        ("Test", X_test, y_test)
    ]:
        overfit_rows.append(evaluate_regression(model, X_part, y_part, name, split_name))

overfit_df = pd.DataFrame(overfit_rows)
display(overfit_df.round(3))

print("Interpretasi awal:")
for name in ["Linear Regression", "Random Forest"]:
    sub = overfit_df[overfit_df["Model"] == name].set_index("Data")
    gap = sub.loc["Test", "RMSE"] - sub.loc["Train", "RMSE"]
    print(f"- {name}: selisih RMSE test - train = {gap:.3f}")

## 10. Interpretasi Fitur

Untuk Random Forest, *feature importance* menunjukkan kontribusi relatif fitur terhadap pembagian pada pohon model. Nilai ini bukan bukti sebab-akibat dan dapat dipengaruhi korelasi antarfitur.

In [ ]:
rf = forest_model.named_steps["model"]
importance = pd.DataFrame({
    "Fitur": features,
    "Feature Importance": rf.feature_importances_
}).sort_values("Feature Importance", ascending=False)

display(importance.round(4))

plt.figure(figsize=(8, 4))
plt.barh(importance["Fitur"][::-1], importance["Feature Importance"][::-1])
plt.title("Feature Importance — Random Forest")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 11. Prediksi untuk Kondisi Gedung Baru

Contoh ini memperkirakan konsumsi energi untuk kondisi operasional tertentu. Hasil merupakan prediksi dari data simulasi, bukan rekomendasi pengelolaan energi yang tervalidasi.

In [ ]:
kondisi_gedung_baru = pd.DataFrame([{
    "suhu_rata_rata_c": 32,
    "jumlah_penghuni": 150,
    "jam_operasional": 11,
    "jumlah_peralatan": 60
}])

prediksi_linear = linear_model.predict(kondisi_gedung_baru)[0]
prediksi_forest = forest_model.predict(kondisi_gedung_baru)[0]

print(f"Prediksi Linear Regression: {prediksi_linear:.2f} kWh/hari")
print(f"Prediksi Random Forest    : {prediksi_forest:.2f} kWh/hari")
print("\nGunakan hasil ini sebagai demonstrasi teknis, bukan dasar tunggal keputusan operasional.")

## 12. Interpretasi Hasil Otomatis

Sel berikut merangkum model dengan RMSE terkecil pada data uji. Model terbaik untuk dataset simulasi ini belum tentu terbaik untuk data nyata atau dataset lain.

In [ ]:
best_row = results_df.iloc[0]
print(f"Model dengan RMSE terkecil pada data uji: {best_row['Model']}")
print(f"MAE  : {best_row['MAE']:.3f} kWh")
print(f"RMSE : {best_row['RMSE']:.3f} kWh")
print(f"R²   : {best_row['R2']:.3f}")

print("\nCara membaca hasil:")
print("- MAE dan RMSE dinyatakan dalam satuan kWh.")
print("- R² menggambarkan kecocokan model terhadap variasi target pada data uji.")
print("- Bandingkan metrik dengan grafik aktual vs prediksi dan selisih train-test.")

## 13. Tantangan, Keterbatasan, dan Etika

1. **Data simulasi:** pola target sengaja dibentuk dari rumus tertentu; kinerja pada data nyata dapat berbeda.
2. **Faktor yang tidak terukur:** jenis gedung, luas bangunan, efisiensi perangkat, cuaca, dan tarif tidak tercakup secara lengkap.
3. **Korelasi bukan kausalitas:** fitur penting tidak otomatis menjadi penyebab konsumsi energi.
4. **Kualitas dan privasi data:** data operasional gedung perlu dikelola secara aman dan sesuai izin.
5. **Akuntabilitas:** prediksi sebaiknya menjadi bahan pendukung, bukan satu-satunya dasar keputusan.
6. **Pemantauan:** model perlu diuji ulang ketika pola pemakaian, peralatan, atau kondisi lingkungan berubah.

Pengembangan lanjutan dapat menggunakan data meter listrik nyata yang diperoleh secara sah, *cross-validation*, fitur waktu/cuaca, dan dokumentasi sumber data.

## 14. Kesimpulan

Notebook ini menerapkan alur dasar *Machine Learning* untuk memprediksi konsumsi energi harian gedung. Tahapan meliputi pembuatan dataset simulasi, pemeriksaan kualitas data, eksplorasi, pembagian data latih dan uji, pelatihan dua algoritma regresi, evaluasi MAE/RMSE/R², pemeriksaan indikasi *overfitting*, interpretasi fitur, dan prediksi data baru.

Implementasi ini mengikuti pokok Introduction to Machine Learning: tujuan prediksi menentukan jenis tugas, kualitas data perlu diperiksa, model harus dievaluasi pada data yang tidak digunakan saat pelatihan, dan hasil perlu ditafsirkan dengan mempertimbangkan keterbatasan serta etika.

**Pengembangan berikutnya:** gunakan dataset nyata yang diperoleh secara sah, lakukan *cross-validation*, bandingkan algoritma tambahan, dan dokumentasikan asumsi analisis.